In [10]:
# Celda 1 — imports
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, PG_ENGINE, BRONZE_DB

In [ ]:
# Celda 2 — función de carga con query explícita
def cargar_a_bronze(nombre_tabla, query):
    print(f"Leyendo {nombre_tabla} desde PostgreSQL...")
    df = pd.read_sql(query, PG_ENGINE)

    df['_ingested_at'] = datetime.now()
    df['_source']      = 'postgresql'
    df = df.where(pd.notnull(df), None)

    CH.execute(f"TRUNCATE TABLE {BRONZE_DB}.raw_{nombre_tabla}")
    CH.execute(f"INSERT INTO {BRONZE_DB}.raw_{nombre_tabla} VALUES", df.to_dict('records'))
    print(f"✓ {len(df):,} filas cargadas en bronze.raw_{nombre_tabla}")

In [17]:
# Celda 3 — ejecutar para todas las tablas
# Celda 3 — ejecutar con SELECT explícito por tabla
tablas = {
    'customers':     "SELECT customer_id, company_name, contact_name, contact_title, address, city, region, postal_code, country, phone, fax FROM customers",
    'orders':        "SELECT order_id, customer_id, employee_id, order_date, required_date, shipped_date, ship_via, freight, ship_name, ship_address, ship_city, ship_region, ship_postal_code, ship_country FROM orders",
    'order_details': "SELECT order_id, product_id, unit_price, quantity, discount FROM order_details",
    'products':      "SELECT product_id, product_name, supplier_id, category_id, quantity_per_unit, unit_price, units_in_stock, units_on_order, reorder_level, discontinued FROM products",
    'categories':    "SELECT category_id, category_name, description FROM categories",
    'suppliers':     "SELECT supplier_id, company_name, contact_name, contact_title, address, city, region, postal_code, country, phone, fax, homepage FROM suppliers",
    'employees': "SELECT employee_id, last_name, first_name, title, title_of_courtesy, birth_date, hire_date, address, city, region, postal_code, country, home_phone, extension, notes, photo_path FROM employees",
    'shippers':      "SELECT shipper_id, company_name, phone FROM shippers",
    'territories':   "SELECT territory_id, territory_description, region_id FROM territories",
    'region':        "SELECT region_id, region_description FROM region",
}

In [18]:
for tabla, query in tablas.items():
    cargar_a_bronze(tabla, query)

Leyendo customers desde PostgreSQL...
✓ 91 filas cargadas en bronze.raw_customers
Leyendo orders desde PostgreSQL...
✓ 830 filas cargadas en bronze.raw_orders
Leyendo order_details desde PostgreSQL...
✓ 2,155 filas cargadas en bronze.raw_order_details
Leyendo products desde PostgreSQL...
✓ 77 filas cargadas en bronze.raw_products
Leyendo categories desde PostgreSQL...
✓ 8 filas cargadas en bronze.raw_categories
Leyendo suppliers desde PostgreSQL...
✓ 29 filas cargadas en bronze.raw_suppliers
Leyendo employees desde PostgreSQL...
✓ 9 filas cargadas en bronze.raw_employees
Leyendo shippers desde PostgreSQL...
✓ 6 filas cargadas en bronze.raw_shippers
Leyendo territories desde PostgreSQL...
✓ 53 filas cargadas en bronze.raw_territories
Leyendo region desde PostgreSQL...
✓ 4 filas cargadas en bronze.raw_region
